# 02 Energía de España

Este dataset es un recopilatorio de datos del mercado eléctrico de España.  
Contiene 6 columnas: `datetime`, `id`, `name`, `geoid`, `geoname` y `value`.  
Preguntas del dataset:
* ¿Qué periodo cubre?
* ¿Cuántas filas tiene?
* ¿Mide solo casas, o qué mide?  
Preguntas:
* ¿Cuándo es más barata la electricidad?
* ¿El consumo cambia conforme cambian las estaciones?
* ¿El consumo cambia dependiendo de la hora del día?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Exploración

In [2]:
df = pd.read_csv("spain_energy_market.csv")
df.head()

,datetime,id,name,geoid,geoname,value
0,2014-01-01 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,25.280833
1,2014-01-02 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,39.924167
2,2014-01-03 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,4.992083
3,2014-01-04 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,4.091667
4,2014-01-05 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,13.587500


In [3]:
df.shape

(40212, 6)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40212 entries, 0 to 40211
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   datetime  40212 non-null  str    
 1   id        40212 non-null  int64  
 2   name      34734 non-null  str    
 3   geoid     10956 non-null  float64
 4   geoname   10956 non-null  str    
 5   value     40212 non-null  float64
dtypes: float64(2), int64(1), str(3)
memory usage: 1.8 MB


In [5]:
df['name'].value_counts()

name
Precio mercado SPOT Diario ESP                                            1826
Precio mercado SPOT Diario FRA                                            1826
Precio mercado SPOT Diario POR                                            1826
Energía asignada en Mercado SPOT Diario España                            1826
Energía asignada en Mercado SPOT Diario Francia                           1826
Generación programada PBF Gas Natural Cogeneración                        1826
Generación programada PBF UGH + no UGH                                    1826
Generación programada PBF Solar fotovoltaica                              1826
Demanda real                                                              1825
Demanda programada PBF total                                              1825
Generación programada PBF total                                           1825
Generación programada PBF Eólica                                          1825
Generación programada PBF Nuclear              

In [6]:
df['datetime'].str[-8:].value_counts()

datetime
22:00:00    23488
23:00:00    16724
Name: count, dtype: int64

In [7]:
print(df['datetime'].min())
print(df['datetime'].max())

2014-01-01 23:00:00
2018-12-31 23:00:00


El dataset tiene 40.212 filas y 6 columnas. Cada fila es el valor diario de una de 20 series del sistema eléctrico (precios del mercado SPOT de España, Francia y Portugal, energía asignada, generación programada por fuente, demanda y rentas de congestión), entre el 1 de enero de 2014 y el 31 de diciembre de 2018. La mayoría de las series tiene un valor por día, unas 1.826 filas.

Como los datos son diarios, no se puede analizar cómo cambia el consumo según la hora del día. `datetime` está guardado como texto, y faltan valores en `name` (5478) y, aún más en `geoid` y `geoname` (29256 en ambas).

## 2. Limpieza y preprocesamiento

In [8]:
df_22 = df[df['datetime'].str[-8:] == '22:00:00']
print(df_22['datetime'].str[5:7].value_counts().sort_index())

print('\ntotal:',df_22['datetime'].str[5:7].value_counts().sum())

datetime
03     518
04    3281
05    3409
06    3261
07    3359
08    3374
09    3302
10    2984
Name: count, dtype: int64

total: 23488


Se confirma que el cambio de hora, de 22:00 a 23:00 viene por el cambio de horario de España, que suele ser de marzo a octubre. Se puede apreciar que 22:00 en el `03` tiene 518, menos que todos porque al final de ese mes se inicia el horario de verano, también se aprecia que en el `10` tiene un poco menos que la normal ya que a final de ese mes también vuelve el horario de invierno.

En este dataset la hora no aporta información, así que de datetime solo nos importa la fecha. Más adelante controlamos que los días coinciden con las fechas.

In [9]:
df_limpio = df.copy()

df_limpio['datetime'] = pd.to_datetime(df_limpio['datetime'])
df_limpio.info()

<class 'pandas.DataFrame'>
RangeIndex: 40212 entries, 0 to 40211
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  40212 non-null  datetime64[us]
 1   id        40212 non-null  int64         
 2   name      34734 non-null  str           
 3   geoid     10956 non-null  float64       
 4   geoname   10956 non-null  str           
 5   value     40212 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(2)
memory usage: 1.8 MB


La columna `datetime` estaba guardada como texto, así que se convirtió a formato fecha `datetime64` en el nuevo DataFrame `df_limpio`.  
Así se puede obtener el día de la semana, el mes o el año, y graficar el tiempo como una línea continua.

In [10]:
df_name_nulo = df_limpio[df_limpio['name'].isnull()]
df_name_nulo['id'].value_counts()

id
600    5478
Name: count, dtype: int64

In [11]:
df_limpio['id'].value_counts()

id
600      10956
602       1826
1334      1826
17        1826
10064     1826
14        1826
1293      1825
10141     1825
10258     1825
10073     1825
4         1825
3         1824
10167     1823
9         1793
1119      1664
1117      1619
1120      1566
1118       712
Name: count, dtype: int64

In [ ]:
df_600 = df_limpio[df_limpio['id'] == 600]
df_600['name'].value_counts()

name
Precio mercado SPOT Diario ESP    1826
Precio mercado SPOT Diario FRA    1826
Precio mercado SPOT Diario POR    1826
Name: count, dtype: int64

In [ ]:
print(df_600[df_600['name'].notnull()]['geoname'].value_counts())
print(df_600[df_600['name'].isnull()]['geoname'].value_counts())

geoname
España      1826
Francia     1826
Portugal    1826
Name: count, dtype: int64
geoname
Portugal    1826
Francia     1826
España      1826
Name: count, dtype: int64


In [ ]:
df_600[df_600['name'].notnull()].head(3)

,datetime,id,name,geoid,geoname,value
0,2014-01-01 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,25.280833
1,2014-01-02 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,39.924167
2,2014-01-03 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,4.992083


In [18]:
df_600[df_600['name'].isnull()].head(3)

,datetime,id,name,geoid,geoname,value
9130,2014-01-01 23:00:00,600,NaN,1.0,Portugal,25.043750
9131,2014-01-01 23:00:00,600,NaN,2.0,Francia,28.711250
9132,2014-01-01 23:00:00,600,NaN,3.0,España,25.280833


In [21]:
df_600.duplicated(subset=['datetime', 'geoname', 'value']).sum()

np.int64(5478)

In [22]:
duplicadas = df_600[df_600.duplicated(subset=['datetime', 'geoname', 'value'])]
duplicadas['name'].isnull().sum()

np.int64(5478)

In [25]:
df_limpio = df_limpio.dropna(subset=['name'])
print(df_limpio.shape)
df_limpio.info()

(34734, 6)
<class 'pandas.DataFrame'>
Index: 34734 entries, 0 to 40211
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  34734 non-null  datetime64[us]
 1   id        34734 non-null  int64         
 2   name      34734 non-null  str           
 3   geoid     5478 non-null   float64       
 4   geoname   5478 non-null   str           
 5   value     34734 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(2)
memory usage: 1.9 MB


Se encontraron 5478 filas que repiten valores de `datetime`, `geoname` y  `value` pero con `name` nulo, todas son copias del precio SPOT de Francia, Portugal y España, y todas están en el `id 600`.

Se eliminaron las copias, que son las que tienen `name` nulo.

In [26]:
df_limpio[df_limpio['geoid'].isnull()]['geoname'].notnull().sum()

np.int64(0)

In [27]:
df_limpio[df_limpio['geoname'].isnull()]['name'].value_counts()

name
Energía asignada en Mercado SPOT Diario España                            1826
Energía asignada en Mercado SPOT Diario Francia                           1826
Generación programada PBF Gas Natural Cogeneración                        1826
Generación programada PBF UGH + no UGH                                    1826
Generación programada PBF Solar fotovoltaica                              1826
Demanda real                                                              1825
Demanda programada PBF total                                              1825
Generación programada PBF total                                           1825
Generación programada PBF Eólica                                          1825
Generación programada PBF Nuclear                                         1825
Generación programada PBF Turbinación bombeo                              1824
Generación programada PBF Carbón                                          1823
Generación programada PBF Ciclo combinado      

In [28]:
df_limpio = df_limpio.drop(columns=['geoid', 'geoname'])
df_limpio.shape

(34734, 4)

Las columnas `geoid` y `geoname` tienen valores nulos en todas las filas que no representan el valor SPOT. Las columnas no aportan ninguna información relevante que la columna `name` no aporte ya. Se tomó la decisión de eliminar las columnas por esta razón.

In [29]:
print(df_limpio.duplicated().sum())
print(df_limpio.duplicated(subset=['datetime', 'name']).sum())

0
0


No se encontraron filas duplicadas, eso significa que cada serie tiene 1 valor por día o menos.

In [30]:
fr_exp = df_limpio[df_limpio['name'] == 'Rentas de congestión mecanismos implícitos diario Francia exportación']
fr_exp['datetime'].dt.month.value_counts().sort_index()

datetime
1      62
2      78
3      95
4      59
5      36
6       4
7       8
8      20
9      43
10    107
11    116
12     84
Name: count, dtype: int64

La serie de menor cantidad de datos, como los 712 de 1826 días en `Rentas de congestión mecanismos implícitos diario Francia exportación` tiene huecos, pero también aparentan seguir un patrón con las estaciones del año, va bajando hacia junio y luego va subiendo a noviembre. Es solo una hipótesis. Pero esta serie no tiene un valor por día, así que no se compara directamente con las series completas.

## 3. Analisis simple

## 4. Analisis complejo